In [1]:
import requests
from collections import deque
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 1 - Default

In [2]:
BASE_URL = "https://quotes.toscrape.com/"

In [ ]:
# All URLS
q = deque([BASE_URL])
visited = set()

i = 0

while q:
    url = q.popleft()
    i += 1
    if i >= 20:
        break
    visited.add(url)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    for ref in soup.find_all("a", href=True):
        next_link = urljoin(response.url, ref["href"])
        if next_link not in visited:
            q.append(next_link)

In [29]:
BASE_URL = "https://quotes.toscrape.com/page/{page_number}/"
page_number = 1
response = requests.get(BASE_URL.format(page_number=page_number))
soup = BeautifulSoup(response.text, "html.parser")
quotes = []

while True:
    for quote in soup.find_all("div", class_="quote"):
        quotes.append(quote.find("span", class_="text").text)

    next_btn = soup.find("li", class_="next")
    if not next_btn:  # Last page reached
        break

    page_number += 1
    soup = BeautifulSoup(
        requests.get(BASE_URL.format(page_number=page_number)).text, "html.parser"
    )

In [ ]:
# CSS Selector

new_page = BASE_URL
quotes = []

while True:
    response = requests.get(new_page)
    soup = BeautifulSoup(response.text, 'html.parser')

    for elem in soup.select('.quote'):
        quotes.append(elem.select('.text')[0].text)

    next_button = soup.select('ul li.next [href]')
    if next_button:
        new_page = urljoin(response.url, next_button[0]['href'])
    else:
        break

# 2 - Scroll

In [19]:
BASE_URL = "https://quotes.toscrape.com/api/quotes?page={pn}"

In [28]:
i = 1
quotes = []
while True:
    response = requests.get(BASE_URL.format(pn=i))

    if len(response.json()['quotes']) == 0:
        break

    for quote in response.json()['quotes']:
        quotes.append(
            quote['text']
        )
    

    i += 1


In [29]:
len(quotes)

100

# 3 - JavaScript

In [2]:
BASE_URL = 'https://quotes.toscrape.com/js/'

In [ ]:
# uv run playwright install webkit
import playwright